## SCD Type 2 via MERGE (production)
Synthetic customer dimension: address changes tracked with `effective_date`/`end_date`/`is_current`.

In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("silver_schema", "lena066636_silver")
dbutils.widgets.text("dimension_table", "customers_scd2")


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

catalog = dbutils.widgets.get("catalog")
dimension_table = f"{catalog}.{dbutils.widgets.get('silver_schema')}.{dbutils.widgets.get('dimension_table')}"


In [0]:
# incoming batch of customer records (source: upstream dimension feed)
updates_df = spark.createDataFrame(
    [(1, "Kyiv", "2026-01-01"), (2, "Lviv", "2026-01-01")],
    ["customer_id", "city", "effective_date"]
).withColumn("effective_date", F.to_date("effective_date"))


In [0]:
if not spark.catalog.tableExists(dimension_table):
    (updates_df
        .withColumn("end_date", F.lit(None).cast("date"))
        .withColumn("is_current", F.lit(True))
        .write.format("delta").saveAsTable(dimension_table))
else:
    target = DeltaTable.forName(spark, dimension_table)
    current_df = target.toDF().filter("is_current = true")

    changed = (updates_df.alias("s")
        .join(current_df.alias("t"), "customer_id")
        .where("t.city <> s.city")
        .select("s.*"))

    staged = (changed.selectExpr("NULL as merge_key", "*")
        .unionByName(updates_df.selectExpr("customer_id as merge_key", "*")))

    (target.alias("t")
        .merge(staged.alias("s"), "t.customer_id = s.merge_key")
        .whenMatchedUpdate(
            condition="t.is_current = true AND t.city <> s.city",
            set={"is_current": "false", "end_date": "s.effective_date"})
        .whenNotMatchedInsert(values={
            "customer_id": "s.customer_id",
            "city": "s.city",
            "effective_date": "s.effective_date",
            "end_date": "CAST(NULL AS DATE)",
            "is_current": "true",
        })
        .execute())
